# 5. Full-scale run on a GPU (Colab / Kaggle)

The committed results were produced on **two CPU threads with no GPU**, and that constraint is visible in their scale: 48 frames, 1400 sequences, 16 epochs, one seed per ablation. This notebook runs the *identical code* at a scale that settles the questions the CPU budget could not.

Nothing here has been run locally. No number in `results/` comes from this notebook.

In [ ]:
# Colab setup. Skip if you already have the repository.
# !git clone https://github.com/HabibaSajid321/skeletal-action-quality-attribution
# %cd skeletal-action-quality-attribution
# !pip install -q torch numpy scipy pandas scikit-learn pyyaml matplotlib
# !pip install -q -e . --no-deps

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt
print('cuda available:', torch.cuda.is_available())

## What to run at full scale, and why

| experiment | CPU budget here | what a GPU settles |
|---|---|---|
| seeds per configuration | 3 | 10+, which is what an accuracy claim needs |
| the 3.0M reference ST-GCN | **not trained** (~40 min/run) | its accuracy, not just its cost |
| sequence length | 48 frames | 300 frames, the ST-GCN paper's regime |
| dataset size | 1400 | 20000, where the data-efficiency curve flattens |

In [ ]:
from saqa.config import load_config
from saqa.pipelines import method_comparison, make_splits
cfg = load_config('configs/base.yaml', [
    'data.num_sequences=20000', 'data.num_frames=192', 'data.batch_size=64',
    'data.num_subjects=120', 'optim.epochs=60',
])
# arms includes the full 3.0M reference architecture, which is out of
# budget on CPU and is the point of running this here.
arms = ('saqa_stgcn', 'stgcn_reference', 'stgcn_reference_large', 'tcn',
        'lstm', 'frame_average')
# table, runs, baselines = method_comparison(cfg, make_splits(cfg), arms)
# table.round(4)

## The real-data path

`saqa.data.real` loads NTU RGB+D skeletons and MTL-AQA / FineDiving style score annotations. Both need a manual download; `scripts/download_real.py` prints the steps and the expected layout. FineDiving is the one public dataset that can validate the *temporal* half of the attribution claim against human annotation rather than against a generator.

In [ ]:
# from saqa.data.real import load_aqa_directory
# coords, scores, ids = load_aqa_directory('data/raw/mtl_aqa', num_frames=192)
# print(coords.shape, scores.min(), scores.max())